In [2]:
import random

DIRTY = '.'   # ô có bụi
CLEAN = ' '   # ô sạch

directions = {"up":    (-1,  0),
              "down":  ( 1,  0),
              "left":  ( 0, -1),
              "right": ( 0,  1),
}

def create_grid():
    grid = [
            [
                DIRTY if random.random() < 0.8 else CLEAN for _ in range(3)
            ]
            for _ in range(3)
    ]
    return grid, (0, 0)

def get_move(robot):
    r, c = robot
    result = []
    for d, (dr, dc) in directions.items():
        nr, nc = r + dr, c + dc
        if 0 <= nr < 3 and 0 <= nc < 3:
            result.append((d, (nr, nc)))
    return result

def print_grid(grid, robot):
    r, c = robot
    print(' ' + '_' * 11)
    for i in range(3):
        cells = []
        for j in range(3):
            if (i, j) == (r, c):
                cells.append(' R ')
            elif grid[i][j] == DIRTY:
                cells.append(' . ')
            else:
                cells.append('   ')
        print('|' + '|'.join(cells) + '|')
        if i < 2:
            print('|---|---|---|')
    print('|___|___|___|')
    dirty_count = sum(grid[i][j] == DIRTY for i in range(3) for j in range(3))
    print(f"\tDirty cells: {dirty_count}")

class VacuumAgent:
    def __init__(self):
        self.state = {
            "position":  None,
            "grid":     [[None]*3 for _ in range(3)],
            "visited":   set(),
            "last_action": None,
        }

        self.rules = [
            {
                "condition": lambda s: all(s["grid"][r][c] == CLEAN
                                           for r in range(3) for c in range(3)),
                "action": "STOP"
            },
            {
                "condition": lambda s: (s["grid"][s["position"][0]][s["position"][1]] == DIRTY),
                "action": "SUCK"
            },
            {
                "condition": lambda s: any(pos not in s["visited"]
                                           for _, pos in get_move(s["position"])),
                "action": "MOVE_UNVISITED"
            },
            {
                "condition": lambda s: any(s["grid"][r][c] == DIRTY
                                           for r in range(3) for c in range(3)
                                           if s["grid"][r][c] is not None),
                "action": "MOVE_TO_DIRTY"
            },
            {
                "condition": lambda s: True,
                "action": "MOVE_RANDOM"
            }
        ]

    def update_state(self, percept, last_action):
        grid, robot = percept
        r, c = robot

        self.state["position"] = robot
        self.state["visited"].add(robot)
        self.state["last_action"] = last_action
        self.state["grid"][r][c] = grid[r][c]

        if last_action == "SUCK":
            self.state["grid"][r][c] = CLEAN

    # RULE-MATCH(state, rules)
    def rule_match(self):
        for rule in self.rules:
            if rule["condition"](self.state):
                return rule
        return None

    def get_action_target(self, action_name):
        pos = self.state["position"]
        neighbors = get_move(pos)
        r, c = pos

        if action_name == "STOP":
            return "STOP", None

        if action_name == "SUCK":
            return "SUCK", pos

        if action_name == "MOVE_UNVISITED":
            unvisited = [(d, p) for d, p in neighbors if p not in self.state["visited"]]
            d, p = random.choice(unvisited)
            return "MOVE", (d, p)

        if action_name == "MOVE_TO_DIRTY":
            best = None
            best_dist = float("inf")
            for nr in range(3):
                for nc in range(3):
                    if self.state["grid"][nr][nc] == DIRTY:
                        dist = abs(nr - r) + abs(nc - c)
                        if dist < best_dist:
                            best_dist = dist
                            best = (nr, nc)
            if best:
                br, bc = best
                for d, (nr, nc) in neighbors:
                    if abs(nr - br) + abs(nc - bc) < best_dist:
                        return "MOVE", (d, (nr, nc))
            # fallback
            d, p = random.choice(neighbors)
            return "MOVE", (d, p)

        # MOVE_RANDOM
        d, p = random.choice(neighbors)
        return "MOVE", (d, p)

    def step(self, percept, last_action):
        self.update_state(percept, last_action)
        matched_rule = self.rule_match()
        action, target = self.get_action_target(matched_rule["action"])
        return action, target

    def print_state(self):
        print("  [ROBOT MEMORY]")
        print('   ' + '_' * 11)
        for i in range(3):
            cells = []
            for j in range(3):
                v = self.state["grid"][i][j]
                if v is None:
                    cells.append(' ? ')
                elif v == DIRTY:
                    cells.append(' . ')
                else:
                    cells.append('   ')
            print('  |' + '|'.join(cells) + '|')
            if i < 2:
                print('  |---|---|---|')
        print('  |___|___|___|')
        print(f"\tVisited : {sorted(self.state['visited'])}")
        print(f"\tPrevious action: {self.state['last_action']}")

grid, robot = create_grid()
agent = VacuumAgent()
last_action = None

print("Initial:")
print_grid(grid, robot)

step = 0
while True:
    percept = (grid, robot)
    action, target = agent.step(percept, last_action)

    print(f"Step {step + 1}: Robot at {robot}")
    print(f"Action : {action}")

    if action == "STOP":
        print("All cells are cleaned — STOP!")
        break

    elif action == "SUCK":
        grid[robot[0]][robot[1]] = CLEAN
        agent.state["grid"][robot[0]][robot[1]] = CLEAN
        last_action = "SUCK"

    elif action == "MOVE":
        d, (nr, nc) = target
        print(f"  Moving {d} to {(nr, nc)}")
        robot = (nr, nc)
        last_action = "MOVE"

    step += 1
    print_grid(grid, robot)
    agent.print_state()

print(f"Completed in {step} steps")
print("Final:")
print_grid(grid, robot)

Initial:
 ___________
| R | . | . |
|---|---|---|
| . |   | . |
|---|---|---|
| . |   | . |
|___|___|___|
	Dirty cells: 7
Step 1: Robot at (0, 0)
Action : SUCK
 ___________
| R | . | . |
|---|---|---|
| . |   | . |
|---|---|---|
| . |   | . |
|___|___|___|
	Dirty cells: 6
  [ROBOT MEMORY]
   ___________
  |   | ? | ? |
  |---|---|---|
  | ? | ? | ? |
  |---|---|---|
  | ? | ? | ? |
  |___|___|___|
	Visited : [(0, 0)]
	Previous action: None
Step 2: Robot at (0, 0)
Action : MOVE
  Moving right to (0, 1)
 ___________
|   | R | . |
|---|---|---|
| . |   | . |
|---|---|---|
| . |   | . |
|___|___|___|
	Dirty cells: 6
  [ROBOT MEMORY]
   ___________
  |   | ? | ? |
  |---|---|---|
  | ? | ? | ? |
  |---|---|---|
  | ? | ? | ? |
  |___|___|___|
	Visited : [(0, 0)]
	Previous action: SUCK
Step 3: Robot at (0, 1)
Action : SUCK
 ___________
|   | R | . |
|---|---|---|
| . |   | . |
|---|---|---|
| . |   | . |
|___|___|___|
	Dirty cells: 5
  [ROBOT MEMORY]
   ___________
  |   |   | ? |
  |---|---